# Day 2 — Chunking, Embeddings & Vector DB Validation

**Module 5 · RAG Testing with RAGAS**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | Chunking strategies | Every strategy trades off differently against the boundary problem |
| 2 | Equivalence partitions for chunking | Same Module 4 Day 4 technique, new dimension (chunk size/overlap instead of prompts) |
| 3 | Reproducing the chunk-boundary bug | Module 4 Day 4 named this bug; today we build it on purpose |
| 4 | `precision@k` / `recall@k` | Hand-computing what RAGAS's `context_precision`/`context_recall` automate |
| 5 | Embedding quality sanity check | Confirm the embedding model does the one thing it's supposed to, before trusting it |
| 6 | Extending the coverage matrix | New columns for retrieval-specific failure modes |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Every cell in this notebook runs offline — no API key needed.

---

> **Where we are in the course**
> Module 4 Day 4, Section 2 named a bug it couldn't build yet:
> *"The chunk-boundary split. This is the single most common real-world bug in RAG systems. A document gets split into 500-token chunks for retrieval. The one sentence that answers the user's question happens to start at token 490 and finish at token 520 — split exactly across two chunks."*
> Today you write the chunking code, reproduce that exact bug on purpose, and fix it — using the same boundary-value-analysis mindset, just pointed at a new boundary.

---
## Chunking strategies

| Strategy | How it splits | Trade-off |
|---|---|---|
| **Fixed-size** | Every N characters/tokens, optionally with overlap | Simple, fast — blind to sentence/paragraph boundaries |
| **Recursive** | Tries paragraph breaks first, falls back to sentences, then characters | Respects document structure better; can still split mid-fact |
| **Semantic** | Splits where embedding similarity between adjacent sentences drops | Best at keeping related facts together; more expensive to compute |

No strategy *eliminates* the boundary problem — each one only changes *how often* a fact gets split, not whether it's possible.

---
## Equivalence partitions for chunking — same technique, new dimension

This is the exact `partitions` dict pattern from Module 4 Day 4's equivalence-partitioning code cell, applied to chunking instead of prompts.

In [8]:
chunking_partitions = {
    "chunk_size": {
        "small":   200,    # more chunks, less context per chunk, fewer boundary splits per fact
        "medium":  500,
        "large":   1500,   # fewer chunks, more context per chunk, but each split is more disruptive
    },
    "overlap": {
        "none":   0,
        "light":  50,
        "heavy":  150,     # higher overlap = lower chance a fact is split across BOTH chunks
    },
    "fact_position": {
        "chunk_middle":   "the answer sentence sits safely inside one chunk",
        "chunk_boundary": "the answer sentence straddles the cut point between two chunks",  # the boundary case
    },
}

for dimension, classes in chunking_partitions.items():
    print(f"=== {dimension} ===")
    for name, value in classes.items():
        print(f"  [{name:<14}] {value}")
    print()

=== chunk_size ===
  [small         ] 200
  [medium        ] 500
  [large         ] 1500

=== overlap ===
  [none          ] 0
  [light         ] 50
  [heavy         ] 150

=== fact_position ===
  [chunk_middle  ] the answer sentence sits safely inside one chunk
  [chunk_boundary] the answer sentence straddles the cut point between two chunks



---
## Reproducing the chunk-boundary bug

A simple fixed-size chunker, and a document engineered so the critical fact sits right at a chunk cut point — the boundary value, deliberately chosen instead of left to chance.

In [10]:
def fixed_size_chunk(text: str, chunk_size: int, overlap: int = 0) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

document = (
    "Our platform was founded in 2019. " + ("Filler sentence about the company history. " * 8) +
    "Refunds are issued within 30 days of purchase, after which only store credit applies. " +
    ("More filler text about unrelated product features. " * 8)
)

fact = "Refunds are issued within 30 days of purchase"
fact_start = document.index(fact)
print(f"Critical fact starts at character {fact_start}")
print()

# Boundary case: chunk_size chosen so the cut lands INSIDE the fact, on purpose.
boundary_chunks = fixed_size_chunk(document, chunk_size=fact_start + 20, overlap=0)
print(f"Chunk 1 ends with   : ...{boundary_chunks[0][-40:]!r}")
print(f"Chunk 2 starts with : {boundary_chunks[1][:40]!r}")
print(f"Fact fully in chunk 1? {fact in boundary_chunks[0]}")
print(f"Fact fully in chunk 2? {fact in boundary_chunks[1]}")
print("-> Neither chunk contains the full fact. A search for 'refund window' will not reliably surface either half.")
print()

# Fix: heavy overlap reduces (does not eliminate) the chance of a split.
fixed_chunks = fixed_size_chunk(document, chunk_size=fact_start + 20, overlap=100)
print(f"With heavy overlap, fact fully in some chunk? {any(fact in c for c in fixed_chunks)}")

Critical fact starts at character 378

Chunk 1 ends with   : ...'he company history. Refunds are issued w'
Chunk 2 starts with : 'ithin 30 days of purchase, after which o'
Fact fully in chunk 1? False
Fact fully in chunk 2? False
-> Neither chunk contains the full fact. A search for 'refund window' will not reliably surface either half.

With heavy overlap, fact fully in some chunk? True


---
## Retrieval correctness: `precision@k` and `recall@k`

This is exactly what RAGAS's `context_precision` and `context_recall` compute for you automatically — using the judge LLM to decide relevance instead of a hand-labeled `relevant_ids` set. Hand-computing it once is what makes the automated version legible instead of a black box.

In [11]:
def precision_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    top_k = retrieved_ids[:k]
    if not top_k:
        return 0.0
    return sum(1 for doc_id in top_k if doc_id in relevant_ids) / len(top_k)

def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    if not relevant_ids:
        return 1.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant_ids) / len(relevant_ids)

# A retriever that found 2 of 3 truly relevant chunks, plus 2 irrelevant ones, in its top 4
retrieved = ["chunk-7", "chunk-2", "chunk-9", "chunk-3"]
relevant  = {"chunk-7", "chunk-3", "chunk-5"}   # chunk-5 was missed entirely

print(f"precision@4 = {precision_at_k(retrieved, relevant, 4):.2f}  (2 of 4 retrieved were relevant)")
print(f"recall@4    = {recall_at_k(retrieved, relevant, 4):.2f}  (2 of 3 relevant chunks were found)")
print()
print("Notice these are independent: a retriever can have perfect precision and weak recall, or vice versa.")

precision@4 = 0.50  (2 of 4 retrieved were relevant)
recall@4    = 0.67  (2 of 3 relevant chunks were found)

Notice these are independent: a retriever can have perfect precision and weak recall, or vice versa.


---
## Embedding quality sanity check

Before trusting an embedding model in production, confirm it does the one thing it's supposed to do: put semantically similar text close together and dissimilar text far apart. We use TF-IDF here as a free, offline stand-in for a real embedding model — and, as you're about to see, that stand-in itself fails part of the sanity check. That failure is the lesson.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "Refunds are issued within 30 days of purchase.",      # 0 — the reference fact
    "Refunds are issued within 45 days of purchase.",      # 1 — lexical near-duplicate (shares almost every word)
    "You can get your money back within a month.",         # 2 — true paraphrase, almost no shared vocabulary
    "Our office is open Monday through Friday.",           # 3 — genuinely unrelated
]

vectors = TfidfVectorizer().fit_transform(sentences)
sims = cosine_similarity(vectors)

print(f"similarity(reference, lexical near-duplicate) = {sims[0][1]:.2f}  (shares most words -> high, as expected)")
print(f"similarity(reference, true paraphrase)         = {sims[0][2]:.2f}  (shares almost no words)")
print(f"similarity(reference, unrelated)                = {sims[0][3]:.2f}  (correctly near zero)")
print()
print("The true paraphrase scores almost as low as the UNRELATED sentence — TF-IDF can't tell them")
print("apart, because it only counts shared words, not shared meaning. A real embedding model is")
print("expected to score the paraphrase meaningfully higher than the unrelated sentence; if it doesn't,")
print("that IS the vector-DB validation failure this sanity check exists to catch.")

similarity(reference, lexical near-duplicate) = 0.81  (shares most words -> high, as expected)
similarity(reference, true paraphrase)         = 0.07  (shares almost no words)
similarity(reference, unrelated)                = 0.00  (correctly near zero)

The true paraphrase scores almost as low as the UNRELATED sentence — TF-IDF can't tell them
apart, because it only counts shared words, not shared meaning. A real embedding model is
expected to score the paraphrase meaningfully higher than the unrelated sentence; if it doesn't,
that IS the vector-DB validation failure this sanity check exists to catch.


---
## Summary

### What we built today
- A concrete chunk-boundary bug, reproduced on purpose using boundary value analysis from Module 4 Day 4
- `precision_at_k()` / `recall_at_k()` — the hand-computed versions of RAGAS's `context_precision`/`context_recall`
- An offline embedding-quality sanity check
- A coverage matrix extended with retrieval-specific failure-mode columns

### The one habit to carry forward
Every new system layer (today: retrieval) gets tested with the *same* equivalence-partitioning, boundary-value, and coverage-matrix process from Module 4 Day 4 — it just gets new partitions and new columns. You are not learning a new testing philosophy every module; you're applying the same one to a new surface.

**Next:** Day 3 — Groundedness, Adversarial Retrieval, Tracing & RAGAS vs DeepEval, where we build hard negatives for the retrieval layer and instrument the whole pipeline with LangSmith.

---